In [435]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [436]:
df=pd.read_csv(r"C:\Users\kirti\Desktop\AI Engineer\End-To-End Project\Movie Recommendation System\data\tmdb_5000_movies.csv")

In [437]:
df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [438]:
df.isnull().sum()

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
dtype: int64

In [439]:
## Droping missing values of overview feature
df.drop([2656, 4140, 4431],axis=0,inplace=True)


In [440]:
df["overview"].isnull().sum()

np.int64(0)

### Overview Feature Cleaning step
- Convert all text to lowercase to eliminate case sensitivity issues.
- Remove punctuation, special characters, and HTML tags using regex (e.g., keep only letters, numbers, spaces). [.,!?]
- Strip extra whitespaces, including leading/trailing spaces and multiple spaces between words.
- Tokenize into words or n-grams, then remove stopwords (common words like "the", "is") using libraries like NLTK.17
- Apply lemmatization or stemming to reduce words to base forms (e.g., "running" → "run").

In [441]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))

def cleaning_overview(string):
    ## Lower case string conversion
    string=string.lower()

    ## Special character removal
    sp_chars=[".",",","!","?" '!', '@', '#', '$', '%', '^', '&', '*', '(', ')', '-', '_', '+', '=', '{', '}', '[', ']', '|', '\\',':', ';', '"', '\'', '<', '>', ',', '.', '?', '/','~', '`']
    for chars in sp_chars:
        string=string.replace(chars,"")
        
    ## Extra spaces removal
    lst=np.array(string.split(" "))

    ## Stopwords removal
    for stword in stop_words:
        for fstrword in lst:
            if stword == fstrword:
                lst[np.where(lst==fstrword)]=''
    final_str=" ".join(lst)
    final_str=" ".join(final_str.split())

    ## Word Tokenizer
    tokens= word_tokenize(final_str)

    ## Stemming
    stemmer=PorterStemmer()
    stemmed_words=[stemmer.stem(words) for words in tokens]

    
    
    return " ".join(stemmed_words)

In [442]:
df["overview"]=df["overview"].apply(cleaning_overview)

In [443]:
data=df[["title","overview"]]
data=data.reset_index(drop=True)

In [444]:
joblib.dump(data,"cleaned_data.pkl")

['cleaned_data.pkl']

In [425]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=5000)
vectors=vectorizer.fit_transform(data["overview"])

In [426]:
import joblib
joblib.dump(vectorizer,"fitted_tfidf.pkl")

['fitted_tfidf.pkl']

In [433]:
joblib.dump(vectors,"movie_vectors.pkl")

['movie_vectors.pkl']

In [429]:
vectors.shape

(4800, 5000)

In [430]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix=cosine_similarity(vectors)

In [431]:
similarity_matrix

array([[1.        , 0.        , 0.        , ..., 0.0397924 , 0.        ,
        0.        ],
       [0.        , 1.        , 0.02705949, ..., 0.0288422 , 0.        ,
        0.        ],
       [0.        , 0.02705949, 1.        , ..., 0.02031242, 0.        ,
        0.        ],
       ...,
       [0.0397924 , 0.0288422 , 0.02031242, ..., 1.        , 0.01148622,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.01148622, 1.        ,
        0.00682867],
       [0.        , 0.        , 0.        , ..., 0.        , 0.00682867,
        1.        ]], shape=(4800, 4800))

In [432]:
joblib.dump(similarity_matrix,"similarity_matrix.pkl")

['similarity_matrix.pkl']

In [403]:
def recommend_function(string):
    data.dropna(inplace=True)
    clean_str=cleaning_overview(string) ## Cleaning function
    idx=data.shape[0]+1
    
    if idx<=data.shape[0]:
        print("If condition")
        data.drop(idx,axis=0,inplace=True)
    else:
        data.loc[idx]=[np.nan,clean_str]
        str_vec=vectorizer.fit_transform(data["overview"])  ## TF-IDF Vectorizer
        sim_matrix=cosine_similarity(str_vec)   ## Cosine Similarity
        recommend_data=pd.Series(sim_matrix[data.shape[0]-1],name="Similarity_score").sort_values(ascending=False)[1:7]
        indexes=recommend_data.index
        scores=recommend_data.values
        final_table=pd.DataFrame(data.iloc[indexes])
        final_table["Confidence Scores"]=scores
    return final_table

In [410]:
fin_table=recommend_function("space mission, black hole, space ship, event horizon")

In [411]:
fin_table

,title,overview,Confidence Scores
1709,Space Pirate Captain Harlock,space pirat captain harlock fearless crew face...,0.318913
2129,The Black Hole,explor craft uss palomino return earth fruitle...,0.318159
1914,Lifeforce,space shuttl mission investig halley comet bri...,0.263322
461,Lost in Space,prospect continu life earth year 2058 grim rob...,0.244658
3533,Galaxina,galaxina lifelik voluptu android assign overse...,0.234562
3328,The Hole,move new neighbourhood brother dane amp luca n...,0.225903
